## CRM-KBOX PROMED REENCODER: PROVIDERS

In [ ]:
#Installs
!pip install xlrd
!pip install openpyxl

In [ ]:
#Imports
import numpy as np
import pandas as pd 
import pycountry
import csv


from datetime import datetime
from contextlib import redirect_stdout #For the log files

In [ ]:
#file_name = "../input/promed/Ventas Regulares Corregido 2.xlsx" 
#df = pd.read_excel(file_name, engine='openpyxl', sheet_name='9ba16c51a1fe738624fe8dd92edc9b4')
file_name = "../input/promed/PROMEDmaestroPROVEEDORES-WORKRECODIFICACIO2.xlsx"
df = pd.read_excel(file_name, sheet_name='RECODIFICACIO2')

In [ ]:
df = df.iloc[:,:7].copy()
df.head()

In [ ]:
df.columns

## 2) Encoder Providers

In [ ]:
##0 - Saving cedula - cli_concat relationships
def cedula_cli_save(table):
    
    ced_un = table['CEDULA'].unique()
    cedula_dict = {}
    
    try:

        for key in ced_un:
            cli_list = []
            key_list = table[(table['CEDULA'] == key)]

            if len(key_list) >= 1:
                          cedula_dict[key] = list(key_list['CONCAT'])

            else: continue
        
        print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Finished cedula - cli saving')
        return cedula_dict
    
    except:
        print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Error during encoding')
        pass


#df = df[df.duplicated(['CEDULA'], keep=False)]
#ced_un = df['CEDULA'].unique()
#cedula_dict = {}
#
#for key in ced_un:
#    cli_list = []
#    for i in range(len(df)): 
#        if key == df.iloc[i,5]:
#            cli_list.append(df.iloc[i,5])
#        else:
#            continue
#    cedula_dict[key] = cli_list 

In [ ]:
saved_dict = cedula_cli_save(df)

In [ ]:
saved_file = open("saved.csv", "w")

writer = csv.writer(saved_file)
for key, value in saved_dict.items():
    writer.writerow([key, value])

saved_file.close()

In [ ]:
len(df['CEDULA'].unique())
df = df.drop_duplicates('CEDULA')

df.head()

In [ ]:
#1 - Countries 
def countryReencoder(table):
    try:
        ## Special conditions
        #conditions = [ table['PAIS'].eq('PA') | table['PAIS'].eq('001'),
        #             table['PAIS'].eq('CR'), 
        #             table['PAIS'].eq(float('nan')), ]
#
        #choices = ['PAN', 'CRI', 'XXX']
#
        #table['countries'] = np.select(conditions, choices, default=table['PAIS'])
        #table['countries'] = table['countries'].fillna('XXX')
        
        table['countries'] = 'XXX'
        
        table['countries'] =  table['countries'].astype('string')

        countries = table['countries'].to_numpy()
        table.drop(['countries',], axis=1, inplace=True)

        print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Finished country encoding')
        return countries
    
    except:
        print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Error during encoding')
        pass

In [ ]:
##1 - Country
countryReencoder(df)

In [ ]:
##2 - Provider Typology
def providerTypeEncoder(providerType = 'XX'):
    
    typeDict = { 'Organismo Público Salud': '110', 'Organismo Públio No Salud': '120',
                'Empresa Privada Salud': '210', 'Empresa Privada No Salud': '220',
               } ## tipus
    
    try:   
        if providerType in typeDict.keys():
            typeCode = typeDict[providerType]
            
            print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Finished provider type encoding. Defined')
            return typeCode
        
        else:
            typeCode = 'XX' # For undefined types
            print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Finished provider type encoding. Undefined')
            return typeCode
         
    except:
        print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Error during encoding')
        pass

In [ ]:
pr = 'Organismo Público Salud'
providerTypeEncoder(pr)

In [ ]:
##3 - Provider Scope/Ambito
def providerScopeEncoder(scope = 'X'):
    
    ##Define hospital sizes
    scopeDict = { 'Nacional_Local': '1', 'Internacional': '2', }
    
    try:
        if scope in scopeDict.keys():
            scopeCode = scopeDict[scope]
            print(datetime.now().strftime("%d-%m-%Y%H:%M:%S"), 'Finished provider scope encoding. Defined')

            return scopeCode
        else:
            scopeCode = 'X' #For undefineds
            
            print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Finished provider scope encoding. Undefined')
            return scopeCode
        
    except:
        print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Error during encoding')
        pass

In [ ]:
sc = 'Nacional_Local'
providerScopeEncoder(sc)

In [ ]:
#4 - Num Secuncial Provider
def providerNumReencoder(table):

    try:
        # Assigning numerical values and storing in another column
        num_client =  table['CEDULA'].astype('category')

        codes = num_client.cat.codes
        cats = num_client.cat.categories
        codesStr = 'P' + codes.astype('string').str.zfill(6)
        
        print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Finished provider num encoding')
        return (codesStr, codes)
    
    except:
        print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Error during encoding')
        pass
        
#from sklearn.preprocessing import LabelEncoder
#le = LabelEncoder()
## Assigning numerical values and storing in another column
#df_s['No Cliente_Cat'] = le.fit_transform(df_s['No Cliente'])
#df_s['No Cliente_Cat']

#import uuid
#uniqueid = uuid.uuid1()


In [ ]:
providerNumReencoder(df)

In [ ]:
def providerEncoder(table):
    try:
        with open('logfile.txt', 'a') as f:
            with redirect_stdout(f):

                table['pCountry'] = countryReencoder(table)
                pType = providerTypeEncoder()
                pSize = providerScopeEncoder()
                table['pNum'] = providerNumReencoder(table)[0]
                
                table['Codigo_KBOX_Proveedor'] = table['pCountry'] + '-' + pType + '-' + pSize + '-' + table['pNum']
                table.drop(['pCountry', 'pNum'], axis=1, inplace=True)
                
                if (table['Codigo_KBOX_Proveedor'].str.len() == 17).all():
                    print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Finished provider complete sequence encoding. Defined')
                    return table['Codigo_KBOX_Proveedor']
                
                else:
                    print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Finished provider complete sequence encoding. Undefined')
                    return table['Codigo_KBOX_Proveedor']
                
    except: 
        with open('logfile.txt', 'a') as f:
            with redirect_stdout(f):
                print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Error during encoding')
        pass  

In [ ]:
providerEncoder(df)

In [ ]:
len(df['NO_PROVE'].unique()), len(df['CEDULA'].unique()), df['CEDULA'].isnull()

In [ ]:
df.to_excel("providers_recoded.xlsx", sheet_name='Recoded', index = False)